# Senior AI Engineer Agent — Qwen3-4B (system prompt, no training)

A lightweight chat agent on **`Qwen/Qwen3-4B-Instruct-2507`** (4B, Apache-2.0, **not gated**).
The "senior AI engineer" persona is a **system prompt** — there is **no dataset / no fine-tuning**.

**Kaggle setup before Run All:**
1. Settings → Accelerator → **GPU T4 x2**
2. *(Optional)* Add-ons → Secrets → `HF_TOKEN` = a valid read token. The model is public, so the token is **optional**; if it's missing/invalid the notebook still runs (login is best-effort and never crashes — this avoids the 401).


In [ ]:
# Optional Hugging Face login. The model is NOT gated, so this is optional.
# Wrapped so a missing/invalid HF_TOKEN secret NEVER crashes the notebook (fixes the 401).
import os
os.environ.setdefault("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") or ""
except Exception:
    pass  # not on Kaggle or no secret attached -> proceed anonymously

if os.environ.get("HF_TOKEN"):
    try:
        from huggingface_hub import login
        login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
        print("Logged in with HF_TOKEN secret.")
    except Exception as e:
        print(f"Token login skipped ({e}). Continuing anonymously (model is public).")
else:
    print("No HF_TOKEN secret found. Continuing anonymously (model is public, not gated).")


In [ ]:
# Install Unsloth (fast 4-bit inference on T4) + Gradio
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q gradio


In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=4096,
    dtype=None,        # auto (T4 -> float16)
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)  # inference speedup

# Silence harmless transformers noise (deprecation FutureWarnings + the
# max_new_tokens/max_length info). max_new_tokens still takes effect.
import warnings, transformers
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=FutureWarning)
# Qwen3-4B generation_config ships max_length=262144 -> clear it so only
# max_new_tokens governs generation (removes the precedence warning).
if getattr(getattr(model, "generation_config", None), "max_length", None):
    model.generation_config.max_length = None
print("Model loaded in 4-bit.")


In [ ]:
# The Senior AI Engineer persona (edit freely to change the agent's behavior)
SYSTEM_PROMPT = """You are a Senior AI/ML Engineer with 10+ years of experience designing, training, evaluating, and shipping production machine-learning systems at scale. You combine deep theoretical understanding with hard-won production experience.

Your expertise spans:
- Architecture of AI systems: LLM apps, RAG, agents, training pipelines, MLOps, model serving.
- Model strategy: when to fine-tune vs. prompt-engineer vs. RAG vs. use an API; cost/latency/accuracy trade-offs.
- Training methodology: pretraining, SFT, RLHF/DPO/GRPO, PEFT (LoRA/QLoRA), data curation, and rigorous evaluation.
- Software engineering: clean modular code, testing, CI/CD, observability, reproducibility.
- Production deployment: quantization, inference optimization, scaling, monitoring, and incident response.

How you operate:
- Reason step by step. Before recommending, weigh cost, latency, accuracy, and maintainability, and state the trade-offs explicitly.
- When requirements are ambiguous, ask 1-2 sharp clarifying questions before committing to a solution.
- Be concrete and practical: give runnable code snippets, specific model/library names, hyperparameter/config values, and concrete next steps. Avoid hand-waving.
- Review critically: proactively surface risks, failure modes, edge cases, and what could break in production.
- Teach the reasoning, not just the answer — briefly explain the WHY behind decisions.
- Be concise and well-structured: short paragraphs, bullet points, and code blocks. No filler or fluff.

Clearly admit when you do not know something or when an approach is experimental/risky. Prioritize correctness and the user's actual goal over sounding impressive."""
print("System prompt set (", len(SYSTEM_PROMPT), "chars ).")


@torch.inference_mode()
def ask(user_msg, history=None, max_new_tokens=1024):
    history = history or []
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for u, a in history:
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": user_msg})
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


In [ ]:
# Inline chat test
reply = ask("I have 50k labeled support tickets. Should I fine-tune a model or build RAG? Walk me through the decision.")
print(reply)


In [ ]:
# Gradio chat UI (share=True is required on Kaggle; public link lasts 72h)
import gradio as gr

def chat(message, history):
    history = [tuple(h) for h in (history or [])]
    return ask(message, history)

demo = gr.ChatInterface(
    fn=chat,
    title="Senior AI Engineer (Qwen3-4B)",
    type="tuples",
    examples=["Design an eval suite for a RAG system.", "Review this training loop for production risks: ..."],
)
demo.launch(share=True)
